# Read MOTEL Ontology (TTL)

This notebook loads the ontology in Python using `rdflib`, then provides:

1. Basic dataset summary
2. Predicate and class inspection
3. Search helpers (keyword / URI fragment)
4. A Mermaid tree visualization of a local neighborhood

In [10]:
from pathlib import Path

import pandas as pd
from rdflib import Graph, RDF, RDFS, OWL, URIRef, Literal

ONTOLOGY_DIR_CANDIDATES = [
    Path("."),
    Path("dev/ontologies"),
    Path("../dev/ontologies"),
    Path("../../dev/ontologies"),
    Path("ontology"),
    Path("../ontology"),
    Path("../../ontology"),
]

ONTOLOGY_DIR = next(
    (
        directory
        for directory in ONTOLOGY_DIR_CANDIDATES
        if directory.exists() and any(directory.glob("*.ttl"))
    ),
    None,
)
if ONTOLOGY_DIR is None:
    raise FileNotFoundError(
        "Could not find any ontology TTL files. Checked: "
        + ", ".join(str(path) for path in ONTOLOGY_DIR_CANDIDATES)
    )

preferred_names = ["motel_ontology.ttl", "motel-ontology.ttl"]
ontology_files = []
for name in preferred_names:
    candidate = ONTOLOGY_DIR / name
    if candidate.exists():
        ontology_files.append(candidate)

ontology_files.extend(
    sorted(
        path
        for path in ONTOLOGY_DIR.glob("*.ttl")
        if path not in ontology_files
    )
)

g = Graph()
for ontology_file in ontology_files:
    g.parse(ontology_file, format="ttl")

print(f"Loaded ontology directory: {ONTOLOGY_DIR}")
print("Loaded TTL files:")
for ontology_file in ontology_files:
    print(f"- {ontology_file}")

print(f"Triples: {len(g):,}")
print(f"Namespaces: {len(list(g.namespaces()))}")


def short_term(term):
    if isinstance(term, URIRef):
        try:
            return g.namespace_manager.normalizeUri(term)
        except Exception:
            return str(term)
    return str(term)

Loaded ontology directory: .
Loaded TTL files:
- motel_ontology.ttl
- motel_attribute.ttl
- motel_carrier.ttl
- motel_process.ttl
- motel_technology.ttl
Triples: 374
Namespaces: 31


In [11]:
# Basic summary tables
classes = set(g.subjects(RDF.type, OWL.Class)) | set(g.subjects(RDF.type, RDFS.Class))
object_properties = set(g.subjects(RDF.type, OWL.ObjectProperty))
datatype_properties = set(g.subjects(RDF.type, OWL.DatatypeProperty))
annotation_properties = set(g.subjects(RDF.type, OWL.AnnotationProperty))
individuals = set(g.subjects(RDF.type, OWL.NamedIndividual))

summary_df = pd.DataFrame(
    [
        ["triples", len(g)],
        ["classes", len(classes)],
        ["object_properties", len(object_properties)],
        ["datatype_properties", len(datatype_properties)],
        ["annotation_properties", len(annotation_properties)],
        ["named_individuals", len(individuals)],
    ],
    columns=["metric", "count"],
)

predicate_counts = (
    pd.Series([short_term(p) for _, p, _ in g], name="predicate")
    .value_counts()
    .rename_axis("predicate")
    .reset_index(name="count")
)

print("Ontology summary")
display(summary_df)
print("Top predicates")
display(predicate_counts.head(20))

Ontology summary


,metric,count
0,triples,374
1,classes,90
2,object_properties,3
3,datatype_properties,5
4,annotation_properties,0
5,named_individuals,0


Top predicates


,predicate,count
0,rdfs:label,102
1,rdf:type,102
2,rdfs:subClassOf,89
3,rdfs:comment,48
4,rdfs:range,8
5,rdfs:domain,8
6,rdf:rest,4
7,owl:versionInfo,4
8,rdf:first,4
9,owl:imports,3


In [12]:
def search_ontology(graph: Graph, query: str, max_results: int = 50, case_sensitive: bool = False) -> pd.DataFrame:
    """Search subject/predicate/object text for a query string."""
    q = query if case_sensitive else query.lower()
    rows = []

    for s, p, o in graph:
        s_txt = short_term(s)
        p_txt = short_term(p)
        o_txt = short_term(o)

        hay = " | ".join([s_txt, p_txt, o_txt])
        hay_cmp = hay if case_sensitive else hay.lower()

        if q in hay_cmp:
            rows.append({"subject": s_txt, "predicate": p_txt, "object": o_txt})
            if len(rows) >= max_results:
                break

    return pd.DataFrame(rows)


# Example search
search_df = search_ontology(g, query="technology", max_results=30)
print(f"Matches: {len(search_df)}")
search_df.head(30)

Matches: 30


,subject,predicate,object
0,motel:ElectrolyserUnit,rdfs:subClassOf,motel:ChemicalSynthesisTechnology
1,motel:StorageTechnology,rdfs:subClassOf,motel:Technology
2,motel:ThermalStorage,rdfs:subClassOf,motel:StorageTechnology
3,motel:FischerTropschUnit,rdfs:subClassOf,motel:ChemicalSynthesisTechnology
4,motel:BoilerSystem,rdfs:subClassOf,motel:ThermalGenerationTechnology
5,motel:ThermalGenerationTechnology,rdfs:label,Thermal Generation Technology
6,motel:CarbonCaptureTechnology,rdf:type,owl:Class
7,motel:Technology,rdfs:subClassOf,motel:MOTELEntity
8,motel:PhotovoltaicSystem,rdfs:subClassOf,motel:ElectricityGenerationTechnology
9,motel:Technology,rdfs:label,Technology


In [17]:
from IPython.display import Markdown, display


def visualize_neighborhood(graph: Graph, center_query: str, max_nodes: int = 40):
    """Visualize a small neighborhood around terms matching center_query as a Mermaid tree."""
    q = center_query.lower()

    def matches_query(term):
        term_text = short_term(term)
        local_name = term_text.split(":", 1)[-1]
        return term_text.lower() == q or local_name.lower() == q

    exact_centers = [s for s in set(graph.subjects()) if matches_query(s)]
    centers = exact_centers or [
        s
        for s in set(graph.subjects())
        if q in short_term(s).lower()
    ]
    centers = centers[:3]

    if not centers:
        print(f"No node found for query: {center_query}")
        return None

    subclass_map = {}
    for child, parent in graph.subject_objects(RDFS.subClassOf):
        subclass_map.setdefault(parent, set()).add(child)

    def node_id(value):
        text = short_term(value)
        cleaned = "".join(ch if ch.isalnum() else "_" for ch in text)
        cleaned = cleaned.strip("_") or "node"
        return cleaned

    def node_label(value):
        return short_term(value).replace("\"", "'")

    lines = ["graph LR"]
    root_id = "root_query"
    lines.append(f'{root_id}["{node_label(center_query)}"]')

    added_nodes = {root_id}
    seen_edges = set()
    remaining = max_nodes

    def add_node(value):
        node = node_id(value)
        if node not in added_nodes:
            lines.append(f'{node}["{node_label(value)}"]')
            added_nodes.add(node)
        return node

    def emit_tree(parent, depth=0):
        nonlocal remaining
        if remaining <= 0:
            return

        children = sorted(subclass_map.get(parent, []), key=short_term)
        for child in children:
            if remaining <= 0:
                break
            edge = (parent, child)
            if edge in seen_edges:
                continue
            seen_edges.add(edge)
            parent_id = add_node(parent)
            child_id = add_node(child)
            lines.append(f"{parent_id} --> {child_id}")
            remaining -= 1
            emit_tree(child, depth + 1)

    for center in centers:
        center_id = add_node(center)
        lines.append(f"{root_id} --> {center_id}")
        emit_tree(center)

    mermaid = "\n".join(lines)
    display(Markdown(f"```mermaid\n{mermaid}\n```"))
    return mermaid


diagram = visualize_neighborhood(g, center_query="Technology", max_nodes=35)
if diagram is not None:
    print(diagram)

```mermaid
graph LR
root_query["Technology"]
motel_Technology["motel:Technology"]
root_query --> motel_Technology
motel_ConversionTechnology["motel:ConversionTechnology"]
motel_Technology --> motel_ConversionTechnology
motel_CarbonCaptureTechnology["motel:CarbonCaptureTechnology"]
motel_ConversionTechnology --> motel_CarbonCaptureTechnology
motel_ChemicalSynthesisTechnology["motel:ChemicalSynthesisTechnology"]
motel_ConversionTechnology --> motel_ChemicalSynthesisTechnology
motel_ElectrolyserUnit["motel:ElectrolyserUnit"]
motel_ChemicalSynthesisTechnology --> motel_ElectrolyserUnit
motel_FischerTropschUnit["motel:FischerTropschUnit"]
motel_ChemicalSynthesisTechnology --> motel_FischerTropschUnit
motel_GasifierUnit["motel:GasifierUnit"]
motel_ChemicalSynthesisTechnology --> motel_GasifierUnit
motel_MethanationSabatierUnit["motel:MethanationSabatierUnit"]
motel_ChemicalSynthesisTechnology --> motel_MethanationSabatierUnit
motel_PyrolyserUnit["motel:PyrolyserUnit"]
motel_ChemicalSynthesisTechnology --> motel_PyrolyserUnit
motel_ElectricityGenerationTechnology["motel:ElectricityGenerationTechnology"]
motel_ConversionTechnology --> motel_ElectricityGenerationTechnology
motel_FuelCell["motel:FuelCell"]
motel_ElectricityGenerationTechnology --> motel_FuelCell
motel_GasTurbine["motel:GasTurbine"]
motel_ElectricityGenerationTechnology --> motel_GasTurbine
motel_HydropowerPlant["motel:HydropowerPlant"]
motel_ElectricityGenerationTechnology --> motel_HydropowerPlant
motel_NuclearReactor["motel:NuclearReactor"]
motel_ElectricityGenerationTechnology --> motel_NuclearReactor
motel_PhotovoltaicSystem["motel:PhotovoltaicSystem"]
motel_ElectricityGenerationTechnology --> motel_PhotovoltaicSystem
motel_WindTurbine["motel:WindTurbine"]
motel_ElectricityGenerationTechnology --> motel_WindTurbine
motel_ThermalGenerationTechnology["motel:ThermalGenerationTechnology"]
motel_ConversionTechnology --> motel_ThermalGenerationTechnology
motel_BoilerSystem["motel:BoilerSystem"]
motel_ThermalGenerationTechnology --> motel_BoilerSystem
motel_HeatPumpUnit["motel:HeatPumpUnit"]
motel_ThermalGenerationTechnology --> motel_HeatPumpUnit
motel_TransmissionInfrastructure["motel:TransmissionInfrastructure"]
motel_ConversionTechnology --> motel_TransmissionInfrastructure
motel_DistrictHeatingNetworkNode["motel:DistrictHeatingNetworkNode"]
motel_TransmissionInfrastructure --> motel_DistrictHeatingNetworkNode
motel_ElectricalGridNode["motel:ElectricalGridNode"]
motel_TransmissionInfrastructure --> motel_ElectricalGridNode
motel_StorageTechnology["motel:StorageTechnology"]
motel_Technology --> motel_StorageTechnology
motel_ElectrochemicalStorage["motel:ElectrochemicalStorage"]
motel_StorageTechnology --> motel_ElectrochemicalStorage
motel_BatteryUnit["motel:BatteryUnit"]
motel_ElectrochemicalStorage --> motel_BatteryUnit
motel_MechanicalStorage["motel:MechanicalStorage"]
motel_StorageTechnology --> motel_MechanicalStorage
motel_ReservoirUnit["motel:ReservoirUnit"]
motel_MechanicalStorage --> motel_ReservoirUnit
motel_ThermalStorage["motel:ThermalStorage"]
motel_StorageTechnology --> motel_ThermalStorage
motel_ThermalStorageTank["motel:ThermalStorageTank"]
motel_ThermalStorage --> motel_ThermalStorageTank
```

graph LR
root_query["Technology"]
motel_Technology["motel:Technology"]
root_query --> motel_Technology
motel_ConversionTechnology["motel:ConversionTechnology"]
motel_Technology --> motel_ConversionTechnology
motel_CarbonCaptureTechnology["motel:CarbonCaptureTechnology"]
motel_ConversionTechnology --> motel_CarbonCaptureTechnology
motel_ChemicalSynthesisTechnology["motel:ChemicalSynthesisTechnology"]
motel_ConversionTechnology --> motel_ChemicalSynthesisTechnology
motel_ElectrolyserUnit["motel:ElectrolyserUnit"]
motel_ChemicalSynthesisTechnology --> motel_ElectrolyserUnit
motel_FischerTropschUnit["motel:FischerTropschUnit"]
motel_ChemicalSynthesisTechnology --> motel_FischerTropschUnit
motel_GasifierUnit["motel:GasifierUnit"]
motel_ChemicalSynthesisTechnology --> motel_GasifierUnit
motel_MethanationSabatierUnit["motel:MethanationSabatierUnit"]
motel_ChemicalSynthesisTechnology --> motel_MethanationSabatierUnit
motel_PyrolyserUnit["motel:PyrolyserUnit"]
motel_ChemicalSynthesisTechnology 